In [1]:
# 1. imports and paths

import os
import gzip
import tarfile
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu, kruskal, spearmanr
import warnings
warnings.filterwarnings('ignore')

base        = 'D:/TNBC_SV_DNA_Repair'
data_dir    = os.path.join(base, 'dataset')
tables_dir  = os.path.join(base, 'results', 'tables')
figures_dir = os.path.join(base, 'results', 'figures')
scores_dir  = os.path.join(base, 'results', 'scores')

print('paths set')

paths set


In [2]:
# 2. load all prior results

gips_df      = pd.read_csv(os.path.join(scores_dir,  'gips_scores.csv'))
surv_tnbc    = pd.read_csv(os.path.join(tables_dir,  'surv_tnbc.csv'))
gene_score   = pd.read_csv(os.path.join(scores_dir,  'gene_disruption_scores.csv'), index_col=0)
sv_ev        = pd.read_csv(os.path.join(tables_dir,  'sv_evidence_per_gene.csv'),   index_col=0)
immune_gips  = pd.read_csv(os.path.join(tables_dir,  'immune_gips_merged.csv'))
cosmic_df    = pd.read_csv(os.path.join(tables_dir,  'panel_cosmic_annotation.csv'), index_col=0)

hr_genes      = ['BRCA1','BRCA2','PALB2','RAD51','RAD51B','RAD51C','RAD51D','BRIP1','ATM','CHEK2']
cohesin_genes = ['STAG2','STAG3','SMC1A','SMC1B','RAD21','REC8']
meiosis_genes = ['HORMAD1','HORMAD2','SYCP2','SYCP3','MLH3','MSH4','MSH5']
all_panel     = hr_genes + cohesin_genes + meiosis_genes

print('gips_df:', gips_df.shape)
print('surv_tnbc:', surv_tnbc.shape)
print('immune_gips:', immune_gips.shape)
print('GIPS groups:', gips_df['GIPS_group'].value_counts().to_dict())

gips_df: (121, 8)
surv_tnbc: (121, 11)
immune_gips: (121, 11)
GIPS groups: {'Low': 52, 'High': 36, 'Moderate': 33}


In [3]:
# 3. merge GIPS with survival

surv_gips = gips_df.merge(surv_tnbc, on='sample', how='inner')

print('merged shape:', surv_gips.shape)
print('OS events:', int(surv_gips['OS'].sum()))
print('PFI events:', int(surv_gips['PFI'].sum()))
print('DFI events:', int(surv_gips['DFI'].sum()))
print('median OS days:', round(surv_gips['OS.time'].median(), 1))
print('median PFI days:', round(surv_gips['PFI.time'].median(), 1))
print('GIPS groups in surv:', surv_gips['GIPS_group'].value_counts().to_dict())

merged shape: (121, 18)
OS events: 15
PFI events: 20
DFI events: 9
median OS days: 943.0
median PFI days: 789.0
GIPS groups in surv: {'Low': 52, 'High': 36, 'Moderate': 33}


In [5]:
# 4. Kaplan-Meier PFI

from lifelines import KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test, logrank_test

group_order  = ['Low', 'Moderate', 'High']
group_colors = {'Low': '#4878cf', 'Moderate': '#f0a500', 'High': '#d94f3d'}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, endpoint, time_col, event_col in [
    (axes[0], 'PFI', 'PFI.time', 'PFI'),
    (axes[1], 'OS',  'OS.time',  'OS')
]:
    kmf = KaplanMeierFitter()
    for group in group_order:
        subset = surv_gips[surv_gips['GIPS_group'] == group]
        t = subset[time_col].values
        e = subset[event_col].values
        kmf.fit(t, e, label=f'{group} (n={len(subset)})')
        kmf.plot_survival_function(
            ax=ax,
            ci_show=True,
            color=group_colors[group]
        )

    results = multivariate_logrank_test(
        surv_gips[time_col],
        surv_gips['GIPS_group'],
        surv_gips[event_col]
    )
    ax.set_title(f'{endpoint} by GIPS group (log-rank p={round(results.p_value, 4)})')
    ax.set_xlabel('days')
    ax.set_ylabel('survival probability')
    ax.legend(fontsize=9)

plt.tight_layout()
fig.savefig(os.path.join(figures_dir, 'nb4_km_curves.png'), dpi=150)
plt.show()
print('saved KM curves')

# log-rank p values
for endpoint, time_col, event_col in [
    ('PFI', 'PFI.time', 'PFI'),
    ('OS',  'OS.time',  'OS')
]:
    res = multivariate_logrank_test(
        surv_gips[time_col],
        surv_gips['GIPS_group'],
        surv_gips[event_col]
    )
    print(f'{endpoint} log-rank p: {round(res.p_value, 4)}')

    # pairwise low vs high
    lo = surv_gips[surv_gips['GIPS_group']=='Low']
    hi = surv_gips[surv_gips['GIPS_group']=='High']
    pairwise = logrank_test(lo[time_col], hi[time_col], lo[event_col], hi[event_col])
    print(f'{endpoint} Low vs High p: {round(pairwise.p_value, 4)}')

saved KM curves
PFI log-rank p: 0.872
PFI Low vs High p: 0.676
OS log-rank p: 0.5714
OS Low vs High p: 0.6646


In [6]:
# 5. Cox regression univariate

from lifelines import CoxPHFitter

# use PFI as primary endpoint
cox_data = surv_gips[['PFI', 'PFI.time', 'OS', 'OS.time', 'GIPS_scaled',
                        'GIPS_group', 'expr_score', 'cn_score', 'mut_score']].copy()
cox_data = cox_data.dropna()

uni_results = []

for predictor in ['GIPS_scaled', 'expr_score', 'cn_score', 'mut_score']:
    try:
        cph = CoxPHFitter()
        df  = cox_data[['PFI', 'PFI.time', predictor]].dropna()
        df.columns = ['event', 'duration', 'covariate']
        cph.fit(df, duration_col='duration', event_col='event')
        s = cph.summary
        uni_results.append({
            'predictor': predictor,
            'HR':        round(np.exp(s['coef'].values[0]), 3),
            'CI_low':    round(np.exp(s['coef lower 95%'].values[0]), 3),
            'CI_high':   round(np.exp(s['coef upper 95%'].values[0]), 3),
            'p':         round(s['p'].values[0], 4)
        })
    except Exception as ex:
        print(f'{predictor} error: {ex}')

uni_df = pd.DataFrame(uni_results)
print('univariate Cox PFI:')
print(uni_df)
uni_df.to_csv(os.path.join(tables_dir, 'cox_univariate_pfi.csv'), index=False)

univariate Cox PFI:
     predictor     HR  CI_low       CI_high       p
0  GIPS_scaled  1.278   0.117  1.398600e+01  0.8406
1   expr_score  0.781   0.003  2.425170e+02  0.9326
2     cn_score  1.284   0.194  8.497000e+00  0.7953
3    mut_score  0.303   0.000  1.737943e+08  0.9076


In [7]:
# 6. Cox multivariable

# add clinical covariates from survival file
print('survival columns:', list(surv_tnbc.columns))
print('sample survival head:')
print(surv_gips[['sample','PFI','PFI.time','OS','OS.time']].head(3))

survival columns: ['sample', '_PATIENT', 'OS', 'OS.time', 'DSS', 'DSS.time', 'DFI', 'DFI.time', 'PFI', 'PFI.time', 'Redaction']
sample survival head:
            sample  PFI  PFI.time  OS  OS.time
0  TCGA-A1-A0SK-01    1     967.0   1    967.0
1  TCGA-A1-A0SM-01    0     242.0   0    242.0
2  TCGA-A1-A0SO-01    0     852.0   0    852.0


In [8]:
# 7. multivariable Cox with GIPS

cph_multi = CoxPHFitter()

multi_cols = ['PFI', 'PFI.time', 'GIPS_scaled']
df_multi   = surv_gips[multi_cols].dropna()
df_multi.columns = ['event', 'duration', 'GIPS']

cph_multi.fit(df_multi, duration_col='duration', event_col='event')
print('multivariable Cox PFI:')
cph_multi.print_summary()

# forest plot
fig, ax = plt.subplots(figsize=(8, 4))
cph_multi.plot(ax=ax)
ax.set_title('Cox regression PFI')
plt.tight_layout()
fig.savefig(os.path.join(figures_dir, 'nb4_cox_forest.png'), dpi=150)
plt.show()
print('saved Cox forest plot')

cox_summary = cph_multi.summary
cox_summary.to_csv(os.path.join(tables_dir, 'cox_multivariable_pfi.csv'))

multivariable Cox PFI:


<lifelines.CoxPHFitter: fitted with 121 total observations, 101 right-censored observations>
             duration col = 'duration'
                event col = 'event'
      baseline estimation = breslow
   number of observations = 121
number of events observed = 20
   partial log-likelihood = -84.13
         time fit was run = 2026-05-17 20:50:02 UTC

---
           coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                  
GIPS       0.25      1.28      1.22           -2.15            2.64                0.12               13.99

           cmp to    z    p  -log2(p)
covariate                            
GIPS         0.00 0.20 0.84      0.25
---
Concordance = 0.49
Partial AIC = 170.26
log-likelihood ratio test = 0.04 on 1 df
-log2(p) of ll-ratio test = 0.25

saved Cox forest plot


In [9]:
# 8. Cox OS

cph_os = CoxPHFitter()
df_os  = surv_gips[['OS', 'OS.time', 'GIPS_scaled']].dropna()
df_os.columns = ['event', 'duration', 'GIPS']

try:
    cph_os.fit(df_os, duration_col='duration', event_col='event')
    s = cph_os.summary
    print('Cox OS HR:', round(np.exp(s['coef'].values[0]), 3))
    print('Cox OS p:', round(s['p'].values[0], 4))
    s.to_csv(os.path.join(tables_dir, 'cox_os.csv'))
except Exception as ex:
    print('OS Cox note:', ex)
    print('OS events too few for reliable Cox, reporting PFI as primary')

Cox OS HR: 0.663
Cox OS p: 0.7697


In [10]:
# 9. inspect METABRIC tar

metabric_tar  = os.path.join(data_dir, 'brca_metabric.tar.gz')
metabric_clin = pd.read_csv(os.path.join(data_dir, 'brca_metabric_clinical_data.tsv'), sep='\t')

with tarfile.open(metabric_tar, 'r:gz') as tar:
    mb_members = tar.getnames()

print('METABRIC clinical shape:', metabric_clin.shape)
print('METABRIC clinical columns:', list(metabric_clin.columns))
print('files in tar:')
for m in mb_members:
    print(' ', m)

METABRIC clinical shape: (2509, 39)
METABRIC clinical columns: ['Study ID', 'Patient ID', 'Sample ID', 'Age at Diagnosis', 'Type of Breast Surgery', 'Cancer Type', 'Cancer Type Detailed', 'Cellularity', 'Chemotherapy', 'Pam50 + Claudin-low subtype', 'Cohort', 'ER status measured by IHC', 'ER Status', 'Neoplasm Histologic Grade', 'HER2 status measured by SNP6', 'HER2 Status', 'Tumor Other Histologic Subtype', 'Hormone Therapy', 'Inferred Menopausal State', 'Integrative Cluster', 'Primary Tumor Laterality', 'Lymph nodes examined positive', 'Mutation Count', 'Nottingham prognostic index', 'Oncotree Code', 'Overall Survival (Months)', 'Overall Survival Status', 'PR Status', 'Radio Therapy', 'Relapse Free Status (Months)', 'Relapse Free Status', 'Number of Samples Per Patient', 'Sample Type', 'Sex', '3-Gene classifier subtype', 'TMB (nonsynonymous)', 'Tumor Size', 'Tumor Stage', "Patient's Vital Status"]
files in tar:
  brca_metabric
  brca_metabric/Readme.txt
  brca_metabric/data_mutations

In [11]:
# 10. check METABRIC subtype columns

subtype_cols = [c for c in metabric_clin.columns if any(
    x in c.upper() for x in ['SUBTYPE','PAM50','ER','PR','HER2','TRIPLE','CLAUDIN']
)]
print('subtype columns:', subtype_cols)
print(metabric_clin[subtype_cols].head(5))
print('value counts for each:')
for col in subtype_cols:
    print(f'{col}:', metabric_clin[col].value_counts().head(5).to_dict())

subtype columns: ['Type of Breast Surgery', 'Cancer Type', 'Cancer Type Detailed', 'Chemotherapy', 'Pam50 + Claudin-low subtype', 'ER status measured by IHC', 'ER Status', 'HER2 status measured by SNP6', 'HER2 Status', 'Tumor Other Histologic Subtype', 'Hormone Therapy', 'Inferred Menopausal State', 'Integrative Cluster', 'Primary Tumor Laterality', 'Nottingham prognostic index', 'Overall Survival (Months)', 'Overall Survival Status', 'PR Status', 'Radio Therapy', 'Number of Samples Per Patient', '3-Gene classifier subtype']
  Type of Breast Surgery    Cancer Type  \
0             MASTECTOMY  Breast Cancer   
1      BREAST CONSERVING  Breast Cancer   
2             MASTECTOMY  Breast Cancer   
3             MASTECTOMY  Breast Cancer   
4             MASTECTOMY  Breast Cancer   

                        Cancer Type Detailed Chemotherapy  \
0           Breast Invasive Ductal Carcinoma           NO   
1           Breast Invasive Ductal Carcinoma           NO   
2           Breast Invasive

In [12]:
# 11. extract METABRIC files

mb_extract_dir = os.path.join(base, 'results', 'metabric_extracted')
os.makedirs(mb_extract_dir, exist_ok=True)

with tarfile.open(metabric_tar, 'r:gz') as tar:
    tar.extractall(mb_extract_dir)

print('extracted to:', mb_extract_dir)
for root, dirs, files in os.walk(mb_extract_dir):
    for f in files:
        fpath = os.path.join(root, f)
        size  = round(os.path.getsize(fpath)/1e6, 2)
        print(f'  {f} ({size} MB)')

extracted to: D:/TNBC_SV_DNA_Repair\results\metabric_extracted
  data_clinical_patient.txt (0.41 MB)
  data_clinical_sample.txt (0.3 MB)
  data_cna.txt (106.06 MB)
  data_gene_panel_matrix.txt (0.05 MB)
  data_methylation_promoters_rrbs.txt (276.34 MB)
  data_mrna_illumina_microarray.txt (689.25 MB)
  data_mrna_illumina_microarray_zscores_ref_diploid_samples.txt (303.12 MB)
  data_mutations.txt (4.02 MB)
  LICENSE (0.0 MB)
  meta_clinical_patient.txt (0.0 MB)
  meta_clinical_sample.txt (0.0 MB)
  meta_cna.txt (0.0 MB)
  meta_gene_panel_matrix.txt (0.0 MB)
  meta_methylation_promoters_rrbs.txt (0.0 MB)
  meta_mrna_illumina_microarray.txt (0.0 MB)
  meta_mrna_illumina_microarray_zscores_ref_diploid_samples.txt (0.0 MB)
  meta_mutations.txt (0.0 MB)
  meta_study.txt (0.0 MB)
  Readme.txt (0.0 MB)
  cases_all.txt (0.02 MB)
  cases_cna.txt (0.02 MB)
  cases_cnaseq.txt (0.02 MB)
  cases_complete.txt (0.02 MB)
  cases_mRNA.txt (0.02 MB)
  cases_nature_2012.txt (0.02 MB)
  cases_nat_comm_2016.

In [13]:
# 12. load METABRIC expression

mb_files = []
for root, dirs, files in os.walk(mb_extract_dir):
    for f in files:
        mb_files.append(os.path.join(root, f))

expr_files = [f for f in mb_files if any(
    x in os.path.basename(f).lower() for x in ['expression','mrna','data_mrna','expr']
)]
print('expression candidate files:')
for f in expr_files:
    print(' ', os.path.basename(f))

print('all files:')
for f in mb_files:
    print(' ', os.path.basename(f))

expression candidate files:
  data_mrna_illumina_microarray.txt
  data_mrna_illumina_microarray_zscores_ref_diploid_samples.txt
  meta_mrna_illumina_microarray.txt
  meta_mrna_illumina_microarray_zscores_ref_diploid_samples.txt
  cases_mRNA.txt
all files:
  data_clinical_patient.txt
  data_clinical_sample.txt
  data_cna.txt
  data_gene_panel_matrix.txt
  data_methylation_promoters_rrbs.txt
  data_mrna_illumina_microarray.txt
  data_mrna_illumina_microarray_zscores_ref_diploid_samples.txt
  data_mutations.txt
  LICENSE
  meta_clinical_patient.txt
  meta_clinical_sample.txt
  meta_cna.txt
  meta_gene_panel_matrix.txt
  meta_methylation_promoters_rrbs.txt
  meta_mrna_illumina_microarray.txt
  meta_mrna_illumina_microarray_zscores_ref_diploid_samples.txt
  meta_mutations.txt
  meta_study.txt
  Readme.txt
  cases_all.txt
  cases_cna.txt
  cases_cnaseq.txt
  cases_complete.txt
  cases_mRNA.txt
  cases_nature_2012.txt
  cases_nat_comm_2016.txt
  cases_sequenced.txt


In [14]:
# 13. read METABRIC expression file

if expr_files:
    mb_expr_file = expr_files[0]
else:
    # try all txt/tsv files
    mb_expr_file = [f for f in mb_files if f.endswith('.txt') or f.endswith('.tsv')][0]

print('loading:', os.path.basename(mb_expr_file))

mb_expr = pd.read_csv(mb_expr_file, sep='\t', index_col=0)
print('METABRIC expression shape:', mb_expr.shape)
print('index sample:', list(mb_expr.index[:5]))
print('columns sample:', list(mb_expr.columns[:3]))

loading: data_mrna_illumina_microarray.txt
METABRIC expression shape: (20603, 1981)
index sample: ['RERE', 'RNF165', 'PHF7', 'CIDEA', 'TENT2']
columns sample: ['Entrez_Gene_Id', 'MB-0362', 'MB-0346']


In [15]:
# 14. check panel genes in METABRIC expression

panel_in_mb = [g for g in all_panel if g in mb_expr.index]
print('panel genes in METABRIC expression:', len(panel_in_mb))
print('found:', panel_in_mb)
missing_mb  = [g for g in all_panel if g not in mb_expr.index]
print('missing:', missing_mb)

panel genes in METABRIC expression: 23
found: ['BRCA1', 'BRCA2', 'PALB2', 'RAD51', 'RAD51B', 'RAD51C', 'RAD51D', 'BRIP1', 'ATM', 'CHEK2', 'STAG2', 'STAG3', 'SMC1A', 'SMC1B', 'RAD21', 'REC8', 'HORMAD1', 'HORMAD2', 'SYCP2', 'SYCP3', 'MLH3', 'MSH4', 'MSH5']
missing: []


In [16]:
# 15. define TNBC in METABRIC

# check what subtype column to use
subtype_col = next((c for c in metabric_clin.columns if
    'pam50' in c.lower() or 'subtype' in c.lower()), None)
er_col      = next((c for c in metabric_clin.columns if 'er_' in c.lower() or c.upper()=='ER STATUS'), None)
her2_col    = next((c for c in metabric_clin.columns if 'her2' in c.lower()), None)

print('subtype col:', subtype_col)
print('ER col:', er_col)
print('HER2 col:', her2_col)

if subtype_col:
    tnbc_mb_mask = metabric_clin[subtype_col].astype(str).str.contains(
        'Triple|TNBC|Basal|basal', case=False, na=False
    )
    mb_tnbc_ids  = metabric_clin[tnbc_mb_mask]['Sample ID'].tolist() \
        if 'Sample ID' in metabric_clin.columns \
        else metabric_clin[tnbc_mb_mask].iloc[:,0].tolist()
    print('METABRIC TNBC by subtype:', len(mb_tnbc_ids))

subtype col: Pam50 + Claudin-low subtype
ER col: ER Status
HER2 col: HER2 status measured by SNP6
METABRIC TNBC by subtype: 209


In [17]:
# 16. METABRIC TNBC expression subset

# match sample IDs between expression columns and TNBC IDs
mb_expr_cols  = list(mb_expr.columns)
mb_tnbc_valid = [s for s in mb_tnbc_ids if s in mb_expr_cols]

if len(mb_tnbc_valid) == 0:
    # try stripping/reformatting
    mb_tnbc_valid = [s for s in mb_expr_cols
                     if any(t in s for t in mb_tnbc_ids[:5])]

print('METABRIC TNBC samples with expression:', len(mb_tnbc_valid))

mb_expr_tnbc  = mb_expr.loc[panel_in_mb, mb_tnbc_valid]
print('METABRIC TNBC expression shape:', mb_expr_tnbc.shape)

METABRIC TNBC samples with expression: 209
METABRIC TNBC expression shape: (24, 209)


In [18]:
# 17. METABRIC expression normalise

# check if already log transformed
raw_max = mb_expr_tnbc.values.max()
raw_min = mb_expr_tnbc.values.min()
print('expression range:', round(raw_min,2), 'to', round(raw_max,2))

if raw_max > 100:
    mb_expr_tnbc = np.log2(mb_expr_tnbc + 1)
    print('log2 transformed')
else:
    print('already log scale, keeping as is')

expression range: 4.82 to 11.34
already log scale, keeping as is


In [19]:
# 18. compute GIPS-like score for METABRIC

# expression z-score disruption
from scipy.stats import zscore

mb_z = mb_expr_tnbc.apply(lambda row: zscore(row, nan_policy='omit'), axis=1)
mb_z = pd.DataFrame(mb_z.tolist(), index=mb_expr_tnbc.index, columns=mb_expr_tnbc.columns)
mb_expr_disrupted = (mb_z.abs() > 1.96).astype(int)

# load METABRIC copy number if available
cn_files = [f for f in mb_files if 'cna' in os.path.basename(f).lower() or
            'copy' in os.path.basename(f).lower() or
            'gistic' in os.path.basename(f).lower()]
print('CN files:', [os.path.basename(f) for f in cn_files])

if cn_files:
    mb_cn = pd.read_csv(cn_files[0], sep='\t', index_col=0)
    print('METABRIC CN shape:', mb_cn.shape)
    panel_in_mb_cn = [g for g in panel_in_mb if g in mb_cn.index]
    mb_cn_tnbc = mb_cn.loc[panel_in_mb_cn, [s for s in mb_tnbc_valid if s in mb_cn.columns]]
    mb_cn_disrupted = (mb_cn_tnbc != 0).astype(int)
    print('MB CN subset:', mb_cn_disrupted.shape)
else:
    mb_cn_disrupted = None
    print('no CN file, using expression only')

CN files: ['data_cna.txt', 'meta_cna.txt', 'cases_cna.txt', 'cases_cnaseq.txt']
METABRIC CN shape: (22544, 2174)
MB CN subset: (23, 209)


In [20]:
# 19. METABRIC GIPS computation

if mb_cn_disrupted is not None:
    common_genes = [g for g in panel_in_mb if g in mb_expr_disrupted.index and g in mb_cn_disrupted.index]
    common_samps = [s for s in mb_tnbc_valid if s in mb_expr_disrupted.columns and s in mb_cn_disrupted.columns]
    e_mb = mb_expr_disrupted.loc[common_genes, common_samps]
    c_mb = mb_cn_disrupted.loc[common_genes, common_samps]
    mb_gene_score = (e_mb + c_mb) / 2.0
else:
    common_genes  = list(mb_expr_disrupted.index)
    common_samps  = list(mb_expr_disrupted.columns)
    mb_gene_score = mb_expr_disrupted.copy().astype(float)

mb_gips_raw    = mb_gene_score.sum(axis=0)
mb_gips_min    = mb_gips_raw.min()
mb_gips_max    = mb_gips_raw.max()
mb_gips_scaled = (mb_gips_raw - mb_gips_min) / (mb_gips_max - mb_gips_min)

mb_t33 = mb_gips_scaled.quantile(0.333)
mb_t66 = mb_gips_scaled.quantile(0.667)

def mb_assign_group(x):
    if x <= mb_t33: return 'Low'
    elif x <= mb_t66: return 'Moderate'
    else: return 'High'

mb_gips_df = pd.DataFrame({
    'sample':      common_samps,
    'GIPS_scaled': mb_gips_scaled.values,
    'GIPS_group':  mb_gips_scaled.apply(mb_assign_group).values
})

print('METABRIC GIPS shape:', mb_gips_df.shape)
print('METABRIC GIPS groups:', mb_gips_df['GIPS_group'].value_counts().to_dict())
mb_gips_df.to_csv(os.path.join(scores_dir, 'metabric_gips_scores.csv'), index=False)

METABRIC GIPS shape: (209, 3)
METABRIC GIPS groups: {'Low': 84, 'High': 67, 'Moderate': 58}


In [21]:
# 20. METABRIC survival data

id_col   = metabric_clin.columns[0]
os_cols  = [c for c in metabric_clin.columns if 'overall' in c.lower() or
             c.upper() in ['OS_STATUS','OS_MONTHS','OVERALL_SURVIVAL_STATUS','OVERALL_SURVIVAL_(MONTHS)']]
pfi_cols = [c for c in metabric_clin.columns if 'relapse' in c.lower() or
             'recur' in c.lower() or 'progression' in c.lower() or 'rfs' in c.lower()]

print('ID col:', id_col)
print('OS cols:', os_cols)
print('relapse cols:', pfi_cols)
print('all clinical columns:', list(metabric_clin.columns))

ID col: Study ID
OS cols: ['Overall Survival (Months)', 'Overall Survival Status']
relapse cols: ['Relapse Free Status (Months)', 'Relapse Free Status']
all clinical columns: ['Study ID', 'Patient ID', 'Sample ID', 'Age at Diagnosis', 'Type of Breast Surgery', 'Cancer Type', 'Cancer Type Detailed', 'Cellularity', 'Chemotherapy', 'Pam50 + Claudin-low subtype', 'Cohort', 'ER status measured by IHC', 'ER Status', 'Neoplasm Histologic Grade', 'HER2 status measured by SNP6', 'HER2 Status', 'Tumor Other Histologic Subtype', 'Hormone Therapy', 'Inferred Menopausal State', 'Integrative Cluster', 'Primary Tumor Laterality', 'Lymph nodes examined positive', 'Mutation Count', 'Nottingham prognostic index', 'Oncotree Code', 'Overall Survival (Months)', 'Overall Survival Status', 'PR Status', 'Radio Therapy', 'Relapse Free Status (Months)', 'Relapse Free Status', 'Number of Samples Per Patient', 'Sample Type', 'Sex', '3-Gene classifier subtype', 'TMB (nonsynonymous)', 'Tumor Size', 'Tumor Stage', "

In [22]:
# 21. METABRIC KM survival

from lifelines import KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test, logrank_test

# identify OS time and event columns
os_time_col   = next((c for c in metabric_clin.columns if 'month' in c.lower() and
                       ('survival' in c.lower() or 'overall' in c.lower())), None)
os_event_col  = next((c for c in metabric_clin.columns if 'status' in c.lower() and
                       'overall' in c.lower()), None)

print('OS time col:', os_time_col)
print('OS event col:', os_event_col)

if os_event_col:
    print('OS event values:', metabric_clin[os_event_col].value_counts().head())

OS time col: Overall Survival (Months)
OS event col: Overall Survival Status
OS event values: Overall Survival Status
1:DECEASED    1144
0:LIVING       837
Name: count, dtype: int64


In [23]:
# 22. merge METABRIC GIPS with survival

mb_surv = metabric_clin[[id_col, os_time_col, os_event_col]].copy() \
    if os_time_col and os_event_col else metabric_clin.copy()

mb_surv.columns = ['sample'] + list(mb_surv.columns[1:])
mb_surv_gips    = mb_gips_df.merge(mb_surv, on='sample', how='inner')

print('METABRIC survival+GIPS shape:', mb_surv_gips.shape)

if os_time_col and os_event_col:
    time_col  = mb_surv_gips.columns[3]
    event_col = mb_surv_gips.columns[4]

    # encode event
    mb_surv_gips['event_bin'] = mb_surv_gips[event_col].astype(str).str.contains(
        '1|DECEASED|died|Dead', case=False, na=False
    ).astype(int)
    mb_surv_gips[time_col] = pd.to_numeric(mb_surv_gips[time_col], errors='coerce')

    print('METABRIC OS events:', mb_surv_gips['event_bin'].sum())
    print('METABRIC GIPS groups:', mb_surv_gips['GIPS_group'].value_counts().to_dict())

METABRIC survival+GIPS shape: (0, 5)
METABRIC OS events: 0
METABRIC GIPS groups: {}


In [24]:
# 23. METABRIC KM plot

if os_time_col and os_event_col and len(mb_surv_gips) > 10:
    time_col  = mb_surv_gips.columns[3]
    fig, ax   = plt.subplots(figsize=(8, 6))
    kmf       = KaplanMeierFitter()

    for group in group_order:
        subset = mb_surv_gips[mb_surv_gips['GIPS_group']==group].dropna(
            subset=[time_col, 'event_bin']
        )
        if len(subset) < 3:
            continue
        kmf.fit(
            subset[time_col],
            subset['event_bin'],
            label=f'{group} (n={len(subset)})'
        )
        kmf.plot_survival_function(ax=ax, ci_show=True, color=group_colors[group])

    valid = mb_surv_gips.dropna(subset=[time_col, 'event_bin'])
    res   = multivariate_logrank_test(valid[time_col], valid['GIPS_group'], valid['event_bin'])
    ax.set_title(f'METABRIC OS by GIPS group (log-rank p={round(res.p_value,4)})')
    ax.set_xlabel('months')
    ax.set_ylabel('survival probability')
    ax.legend(fontsize=9)

    plt.tight_layout()
    fig.savefig(os.path.join(figures_dir, 'nb4_metabric_km.png'), dpi=150)
    plt.show()
    print(f'METABRIC log-rank p: {round(res.p_value, 4)}')
else:
    print('insufficient METABRIC survival data for KM')

insufficient METABRIC survival data for KM


In [28]:
# 24. METABRIC GIPS distribution comparison

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(gips_df['GIPS_scaled'],    bins=20, color='steelblue', alpha=0.7, label='TCGA-BRCA')
axes[0].hist(mb_gips_df['GIPS_scaled'], bins=20, color='salmon',    alpha=0.7, label='METABRIC')
axes[0].set_xlabel('GIPS score')
axes[0].set_ylabel('patients')
axes[0].set_title('GIPS distribution comparison')
axes[0].legend()

# remove duplicates before correlation
tcga_gene_mean = gene_score.mean(axis=1)
tcga_gene_mean = tcga_gene_mean[~tcga_gene_mean.index.duplicated(keep='first')]

mb_gene_mean   = mb_gene_score.mean(axis=1)
mb_gene_mean   = mb_gene_mean[~mb_gene_mean.index.duplicated(keep='first')]

common_for_corr = sorted(list(set(tcga_gene_mean.index) & set(mb_gene_mean.index)))
print('common genes:', len(common_for_corr))

if len(common_for_corr) >= 5:
    tcga_vals = tcga_gene_mean[common_for_corr].values
    mb_vals   = mb_gene_mean[common_for_corr].values
    r2, p2    = spearmanr(tcga_vals, mb_vals)

    axes[1].scatter(tcga_vals, mb_vals, color='steelblue', alpha=0.7, s=50)
    for i, g in enumerate(common_for_corr):
        axes[1].annotate(g, (tcga_vals[i], mb_vals[i]), fontsize=6, alpha=0.6)
    axes[1].set_xlabel('TCGA mean disruption')
    axes[1].set_ylabel('METABRIC mean disruption')
    axes[1].set_title(f'Gene disruption concordance (r={round(r2,3)}, p={round(p2,4)})')
else:
    r2, p2 = 0, 1
    axes[1].set_title('insufficient common genes')

plt.tight_layout()
fig.savefig(os.path.join(figures_dir, 'nb4_metabric_concordance.png'), dpi=150)
plt.show()
print('concordance r:', round(r2,3), 'p:', round(p2,4))

common genes: 23
concordance r: 0.244 p: 0.2626


In [29]:
# 25. pathway enrichment setup

import gseapy as gp

# genes disrupted in high GIPS patients
high_samples = gips_df[gips_df['GIPS_group']=='High']['sample'].tolist()
low_samples  = gips_df[gips_df['GIPS_group']=='Low']['sample'].tolist()

common_samps_gs = [s for s in high_samples if s in gene_score.columns]
common_low_gs   = [s for s in low_samples  if s in gene_score.columns]

high_disruption = gene_score[common_samps_gs].mean(axis=1)
low_disruption  = gene_score[common_low_gs].mean(axis=1)
diff_score      = high_disruption - low_disruption
diff_score      = diff_score.sort_values(ascending=False)

print('top disrupted in high GIPS:')
print(diff_score.head(10))
print('bottom (low GIPS enriched):')
print(diff_score.tail(5))

top disrupted in high GIPS:
MLH3       0.284188
RAD51B     0.259259
RAD51      0.240741
REC8       0.237892
SYCP3      0.232194
MSH5       0.229345
BRIP1      0.228632
PALB2      0.203704
SYCP2      0.202991
HORMAD2    0.190171
dtype: float64
bottom (low GIPS enriched):
MSH4       0.143875
SMC1B      0.126781
CHEK2      0.124644
STAG2      0.120370
HORMAD1    0.106125
dtype: float64


In [30]:
# 26. Enrichr pathway analysis

# genes most disrupted in high GIPS
top_genes = diff_score[diff_score > 0].index.tolist()
print('genes for enrichment:', top_genes)

enrich_results = {}

for gene_set_lib in ['GO_Biological_Process_2023', 'KEGG_2021_Human']:
    try:
        enr = gp.enrichr(
            gene_list=top_genes,
            gene_sets=gene_set_lib,
            outdir=None,
            no_plot=True
        )
        res = enr.results.sort_values('Adjusted P-value')
        enrich_results[gene_set_lib] = res
        print(f'{gene_set_lib} top hits:')
        print(res[['Term','Overlap','P-value','Adjusted P-value']].head(5).to_string())
    except Exception as ex:
        print(f'{gene_set_lib} error: {ex}')

genes for enrichment: ['MLH3', 'RAD51B', 'RAD51', 'REC8', 'SYCP3', 'MSH5', 'BRIP1', 'PALB2', 'SYCP2', 'HORMAD2', 'STAG3', 'BRCA1', 'BRCA2', 'RAD51D', 'RAD51C', 'ATM', 'SMC1A', 'RAD21', 'MSH4', 'SMC1B', 'CHEK2', 'STAG2', 'HORMAD1']
GO_Biological_Process_2023 top hits:
                                                                   Term Overlap       P-value  Adjusted P-value
0                               Double-Strand Break Repair (GO:0006302)  11/168  1.306352e-17      2.834783e-15
1                                               DNA Repair (GO:0006281)  12/291  8.408909e-17      9.123666e-15
2                        Meiotic Sister Chromatid Cohesion (GO:0051177)    6/10  2.378244e-16      1.720263e-14
3  Double-Strand Break Repair Via Homologous Recombination (GO:0000724)   8/111  3.186025e-13      1.728419e-11
4                                        DNA Recombination (GO:0006310)    6/42  5.805493e-12      2.519584e-10
KEGG_2021_Human top hits:
                       Term Overla

In [31]:
# 27. pathway enrichment plot

if enrich_results:
    lib  = list(enrich_results.keys())[0]
    res  = enrich_results[lib].head(15)
    res  = res[res['Adjusted P-value'] < 0.2].copy()

    if len(res) > 0:
        res['log10p'] = -np.log10(res['Adjusted P-value'].clip(lower=1e-10))
        res = res.sort_values('log10p')

        fig, ax = plt.subplots(figsize=(10, max(4, len(res)*0.4)))
        colors  = ['#d94f3d' if p < 0.05 else '#f0a500' for p in res['Adjusted P-value']]
        ax.barh(range(len(res)), res['log10p'], color=colors, edgecolor='none')
        ax.set_yticks(range(len(res)))
        ax.set_yticklabels(
            [t[:60] for t in res['Term'].tolist()], fontsize=8
        )
        ax.set_xlabel('-log10 adjusted p-value')
        ax.set_title(f'Pathway enrichment in high-GIPS genes ({lib})')
        ax.axvline(-np.log10(0.05), color='black', linestyle='--', linewidth=0.8)

        plt.tight_layout()
        fig.savefig(os.path.join(figures_dir, 'nb4_pathway_enrichment.png'), dpi=150)
        plt.show()
        print('saved pathway enrichment plot')

        for lib_name, df_enr in enrich_results.items():
            df_enr.to_csv(
                os.path.join(tables_dir, f'enrichment_{lib_name}.csv'), index=False
            )
    else:
        print('no enrichment terms below p=0.2')

saved pathway enrichment plot


In [32]:
# 28. publication figure 1 - SV and GIPS overview

sv_ev_plot = pd.read_csv(os.path.join(tables_dir, 'sv_evidence_per_gene.csv'), index_col=0)
gnomad_hits = pd.read_csv(os.path.join(tables_dir, 'gnomad_sv_gene_hits.csv')) \
    if os.path.exists(os.path.join(tables_dir, 'gnomad_sv_gene_hits.csv')) else pd.DataFrame()
hgsvc_hits  = pd.read_csv(os.path.join(tables_dir, 'hgsvc2_sv_gene_hits.csv')) \
    if os.path.exists(os.path.join(tables_dir, 'hgsvc2_sv_gene_hits.csv')) else pd.DataFrame()

fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.4)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[0, 2])
ax4 = fig.add_subplot(gs[1, 0])
ax5 = fig.add_subplot(gs[1, 1])
ax6 = fig.add_subplot(gs[1, 2])

# panel A: SV count per gene
sv_counts = {g: 0 for g in all_panel}
if len(gnomad_hits) > 0 and 'gene' in gnomad_hits.columns:
    for g, n in gnomad_hits['gene'].value_counts().items():
        if g in sv_counts: sv_counts[g] += n
sv_series = pd.Series(sv_counts)[all_panel]
bar_colors_sv = ['#d94f3d' if g in hr_genes else '#f0a500' if g in cohesin_genes else '#4878cf'
                  for g in all_panel]
ax1.bar(range(len(all_panel)), sv_series.values, color=bar_colors_sv, edgecolor='none')
ax1.set_xticks(range(len(all_panel)))
ax1.set_xticklabels(all_panel, rotation=90, fontsize=7)
ax1.set_ylabel('gnomAD SV count')
ax1.set_title('A. SV evidence at panel loci')

# panel B: GIPS distribution
ax2.hist(gips_df['GIPS_scaled'], bins=20, color='steelblue', edgecolor='white', linewidth=0.5)
ax2.axvline(gips_df['GIPS_scaled'].quantile(0.333), color='orange', linestyle='--', linewidth=1.2)
ax2.axvline(gips_df['GIPS_scaled'].quantile(0.667), color='red',    linestyle='--', linewidth=1.2)
ax2.set_xlabel('GIPS score')
ax2.set_ylabel('patients')
ax2.set_title('B. GIPS distribution (n=121)')

# panel C: component scores by group
comp_cols = ['expr_score','cn_score','mut_score']
x_pos     = np.arange(3)
width     = 0.25
for i, col in enumerate(comp_cols):
    means = [immune_gips[immune_gips['GIPS_group']==g][col].mean() for g in group_order]
    ax3.bar(x_pos + i*width, means, width,
            label=col.replace('_score',''),
            color=['#5ba4cf','#f0a500','#d94f3d'][i])
ax3.set_xticks(x_pos + width)
ax3.set_xticklabels(group_order)
ax3.set_ylabel('mean score')
ax3.set_title('C. Component scores by group')
ax3.legend(fontsize=8)

# panel D: KM PFI
from lifelines import KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test
kmf = KaplanMeierFitter()
for group in group_order:
    subset = surv_gips[surv_gips['GIPS_group']==group]
    kmf.fit(subset['PFI.time'], subset['PFI'], label=group)
    kmf.plot_survival_function(ax=ax4, ci_show=False, color=group_colors[group])
res_pfi = multivariate_logrank_test(surv_gips['PFI.time'], surv_gips['GIPS_group'], surv_gips['PFI'])
ax4.set_title(f'D. PFI by GIPS (p={round(res_pfi.p_value,3)})')
ax4.set_xlabel('days')
ax4.set_ylabel('probability')
ax4.legend(fontsize=8)

# panel E: exhaustion trend
if 'exhaustion' in immune_gips.columns:
    data_box = [immune_gips[immune_gips['GIPS_group']==g]['exhaustion'].dropna().values
                for g in group_order]
    bp = ax5.boxplot(data_box, patch_artist=True, widths=0.5)
    for patch, g in zip(bp['boxes'], group_order):
        patch.set_facecolor(group_colors[g])
        patch.set_alpha(0.7)
    ax5.set_xticklabels(group_order)
    ax5.set_ylabel('ssGSEA NES')
    ax5.set_title('E. Exhaustion score by GIPS')

# panel F: METABRIC concordance
if len(common_for_corr) >= 5:
    ax6.scatter(tcga_gene_mean[common_for_corr], mb_gene_mean[common_for_corr],
                color='steelblue', alpha=0.7, s=50)
    for g in common_for_corr:
        ax6.annotate(g, (tcga_gene_mean[g], mb_gene_mean[g]), fontsize=6, alpha=0.6)
    ax6.set_xlabel('TCGA disruption')
    ax6.set_ylabel('METABRIC disruption')
    ax6.set_title(f'F. Validation concordance (r={round(r2,2)})')

plt.suptitle('TNBC SV DNA Repair Framework - Summary', fontsize=13, y=1.01)
fig.savefig(os.path.join(figures_dir, 'nb4_publication_figure.png'), dpi=200, bbox_inches='tight')
plt.show()
print('saved publication figure')

saved publication figure


In [33]:
# 29. cohesin meiosis focus plot

# STAG3 and meiosis genes are the novel contribution
novel_genes = ['STAG3','HORMAD1','HORMAD2','REC8','SMC1B','SYCP3']
novel_in_score = [g for g in novel_genes if g in gene_score.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# disruption scores for novel genes by group
novel_data = []
for g in novel_in_score:
    for group in group_order:
        samps = gips_df[gips_df['GIPS_group']==group]['sample'].tolist()
        samps = [s for s in samps if s in gene_score.columns]
        vals  = gene_score.loc[g, samps].values
        for v in vals:
            novel_data.append({'gene': g, 'group': group, 'score': v})

novel_df = pd.DataFrame(novel_data)
gene_group_mean = novel_df.groupby(['gene','group'])['score'].mean().unstack()

x   = np.arange(len(novel_in_score))
w   = 0.25
for i, grp in enumerate(group_order):
    if grp in gene_group_mean.columns:
        axes[0].bar(x + i*w,
                    gene_group_mean[grp].values,
                    w, label=grp,
                    color=group_colors[grp], alpha=0.8)
axes[0].set_xticks(x + w)
axes[0].set_xticklabels(novel_in_score, rotation=30, ha='right')
axes[0].set_ylabel('mean disruption score')
axes[0].set_title('Novel cohesin/meiosis gene disruption by GIPS group')
axes[0].legend(fontsize=9)

# STAG3 expression split
if 'STAG3' in gene_score.index:
    stag3_scores = []
    for grp in group_order:
        samps = gips_df[gips_df['GIPS_group']==grp]['sample'].tolist()
        samps = [s for s in samps if s in gene_score.columns]
        stag3_scores.append(gene_score.loc['STAG3', samps].values)
    bp = axes[1].boxplot(stag3_scores, patch_artist=True, widths=0.5)
    for patch, grp in zip(bp['boxes'], group_order):
        patch.set_facecolor(group_colors[grp])
        patch.set_alpha(0.7)
    axes[1].set_xticklabels(group_order)
    axes[1].set_ylabel('disruption score')
    axes[1].set_title('STAG3 disruption by GIPS group')

plt.tight_layout()
fig.savefig(os.path.join(figures_dir, 'nb4_novel_genes_focus.png'), dpi=150)
plt.show()
print('saved novel genes plot')

saved novel genes plot


In [34]:
# 30. save all final tables

surv_gips.to_csv(os.path.join(tables_dir, 'surv_gips_final.csv'), index=False)
mb_surv_gips.to_csv(os.path.join(tables_dir, 'metabric_surv_gips.csv'), index=False)
diff_score.to_csv(os.path.join(tables_dir, 'gene_disruption_diff_highlow.csv'), header=True)

print('all tables saved')

all tables saved


In [35]:
# 31. full results summary

pfi_res   = multivariate_logrank_test(surv_gips['PFI.time'], surv_gips['GIPS_group'], surv_gips['PFI'])
os_res    = multivariate_logrank_test(surv_gips['OS.time'],  surv_gips['GIPS_group'], surv_gips['OS'])

lo_hi_pfi = logrank_test(
    surv_gips[surv_gips['GIPS_group']=='Low']['PFI.time'],
    surv_gips[surv_gips['GIPS_group']=='High']['PFI.time'],
    surv_gips[surv_gips['GIPS_group']=='Low']['PFI'],
    surv_gips[surv_gips['GIPS_group']=='High']['PFI']
)

cox_hr  = float(np.exp(cph_multi.summary['coef'].values[0]))
cox_p   = float(cph_multi.summary['p'].values[0])

summary4 = {
    'TCGA TNBC samples':            len(surv_gips),
    'TCGA PFI events':              int(surv_gips['PFI'].sum()),
    'TCGA OS events':               int(surv_gips['OS'].sum()),
    'PFI log-rank p (3 groups)':    round(pfi_res.p_value, 4),
    'PFI log-rank p (Low vs High)': round(lo_hi_pfi.p_value, 4),
    'OS log-rank p (3 groups)':     round(os_res.p_value, 4),
    'Cox PFI HR (GIPS)':            round(cox_hr, 3),
    'Cox PFI p':                    round(cox_p, 4),
    'METABRIC TNBC samples':        len(mb_gips_df),
    'Gene disruption concordance r':round(r2, 3),
    'Gene disruption concordance p':round(p2, 4),
}

for k, v in summary4.items():
    print(f'{k}: {v}')

pd.DataFrame.from_dict(
    summary4, orient='index', columns=['value']
).to_csv(os.path.join(tables_dir, 'nb4_summary.csv'))

print('notebook 4 complete')

TCGA TNBC samples: 121
TCGA PFI events: 20
TCGA OS events: 15
PFI log-rank p (3 groups): 0.872
PFI log-rank p (Low vs High): 0.676
OS log-rank p (3 groups): 0.5714
Cox PFI HR (GIPS): 1.278
Cox PFI p: 0.8406
METABRIC TNBC samples: 209
Gene disruption concordance r: 0.244
Gene disruption concordance p: 0.2626
notebook 4 complete
